<a href="https://www.kaggle.com/code/nswitzer/eval-unsloth-who-fine-tune-vs-vanilla-gemma-4?scriptVersionId=320441284" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Eval: Unsloth Fine-tune vs Vanilla Gemma 4 — WHO IMGS Medical Q&A

Compares `gemma4:e2b` (vanilla) against `hf.co/nswitzer/gemma4-maritime-medical-GGUF` (Unsloth fine-tune on the WHO International Medical Guide for Ships) on:

- **20 held-out medical questions** — pharmacology, burns, cardiac, tropical medicine, obstetrics, poisoning, hypothermia, etc. (drawn from WHO IMGS pages not in the training set)
- **10 general maritime questions** — SOLAS, engineering, navigation (regression check: both models should score similarly here)

**Judge:** `gemma-4-26b-a4b-it` via Google AI Studio — Gemma judging Gemma keeps the eval on-brand for the Gemma hackathon. Scoring on four 1–10 axes: accuracy, specificity, citation quality (WHO IMGS page references), and anti-hallucination.

**Requirements:**
- Ollama running with both models pulled (see setup cell)
- `GOOGLE_API_KEY` Kaggle secret for the judge

In [1]:
%%capture
!pip install httpx ollama

In [2]:
import subprocess, time, os

# Kaggle's base image lacks zstd, which the Ollama installer needs to unpack.
print('Installing zstd...')
subprocess.run('apt-get update -qq && apt-get install -y -qq zstd', shell=True, check=True)

# Install Ollama
print('Installing Ollama...')
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

# Start Ollama server in background
print('Starting Ollama server...')
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Poll until Ollama is accepting connections — T4 cold-start can take 15-20s,
# a fixed sleep(5) is not enough.
import urllib.request
deadline = time.time() + 60
while time.time() < deadline:
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        print('Ollama ready.')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not start within 60 seconds')

# Pull both models (Kaggle has fast internet)
BASELINE_MODEL = 'gemma4:e2b'
MEDICAL_MODEL  = 'hf.co/nswitzer/gemma4-maritime-medical-GGUF'

for model in [BASELINE_MODEL, MEDICAL_MODEL]:
    print(f'Pulling {model}...')
    subprocess.run(['ollama', 'pull', model], check=True)
    print(f'  {model} ready')

print('\nAll models ready.')

Installing zstd...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Installing Ollama...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Starting Ollama server...
Ollama ready.
Pulling gemma4:e2b...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 4e30e2665218:   0% ▕                  ▏  12 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   1% ▕                  ▏  95 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   2% ▕                  ▏ 146 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   3% ▕                  ▏ 233 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   5% ▕                  ▏ 331 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   5% ▕                  ▏ 381 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   7% ▕█                 ▏ 478 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   8% ▕█                 ▏ 579 MB/7.2 GB                  pulling manifest 
pulling 4e30e2665218:   9% ▕█                 ▏ 626 MB

  gemma4:e2b ready
Pulling hf.co/nswitzer/gemma4-maritime-medical-GGUF...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 45563d9a0204:   0% ▕                  ▏ 403 KB/3.4 GB                  pulling manifest 
pulling 45563d9a0204:   0% ▕                  ▏  16 MB/3.4 GB                  pulling manifest 
pulling 45563d9a0204:   2% ▕                  ▏  52 MB/3.4 GB                  pulling manifest 
pulling 45563d9a0204:   2% ▕                  ▏  72 MB/3.4 GB                  pulling manifest 
pulling 45563d9a0204:   3% ▕                  ▏ 112 MB/3.4 GB                  pulling manifest 
pulling 45563d9a0204:   4% ▕                  ▏ 131 MB/3.4 GB                  pulling manifest 
pulling 45563d9a0204:   5% ▕                  ▏ 170 MB/3.4 GB                  pulling manifest 
pulling 45563d9a0204:   6% ▕█                 ▏ 208 MB/3.4 GB                  pulling manifest 
pulling 45563d9a0204:   7% ▕█                 ▏ 226 MB/3.4 GB                  pulling manifest 
pulling 45563d

  hf.co/nswitzer/gemma4-maritime-medical-GGUF ready

All models ready.


pulling manifest 
pulling 45563d9a0204: 100% ▕██████████████████▏ 3.4 GB                         
pulling f56e8459650d: 100% ▕██████████████████▏  159 B                         
pulling f5107f3ab6b0: 100% ▕██████████████████▏   52 B                         
pulling ab4b1a755020: 100% ▕██████████████████▏  555 B                         
verifying sha256 digest 
writing manifest 
success 


In [3]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GOOGLE_API_KEY = secrets.get_secret('GOOGLE_API_KEY')
print('API key loaded:', bool(GOOGLE_API_KEY))

API key loaded: True


In [4]:
MEDICAL_QUESTIONS = [
    {'question': 'What is the recommended adult oral dose of amoxicillin for a chest infection at sea, and how long should treatment continue?', 'domain': 'medical', 'topic': 'pharmacology'},
    {'question': 'A sailor has been stung by a jellyfish and is experiencing urticaria. What antihistamine is available on the ship\'s medicine chest and what dose should be given?', 'domain': 'medical', 'topic': 'envenomation'},
    {'question': 'The MPIC needs to give an intramuscular injection. Describe the correct site and technique to avoid hitting the sciatic nerve.', 'domain': 'medical', 'topic': 'procedures'},
    {'question': 'A crew member has been prescribed metronidazole. What food interaction must they be warned about?', 'domain': 'medical', 'topic': 'pharmacology'},
    {'question': 'What are the WHO IMGS criteria for deciding to evacuate a patient with a suspected appendicitis versus managing on board?', 'domain': 'medical', 'topic': 'surgical emergencies'},
    {'question': 'How do you estimate the percentage body surface area burned using the Rule of Nines? Give the percentage for the head, each arm, each leg, and the torso.', 'domain': 'medical', 'topic': 'burns'},
    {'question': 'A crew member has a deep second-degree burn covering approximately 20% BSA. What is the fluid resuscitation formula and how much fluid should be given in the first 8 hours?', 'domain': 'medical', 'topic': 'burns'},
    {'question': 'Describe the WHO IMGS approach to wound closure for a laceration on the scalp. When is it appropriate to close with adhesive strips vs sutures?', 'domain': 'medical', 'topic': 'wounds'},
    {'question': 'A 52-year-old Chief Engineer has crushing central chest pain radiating to the left arm. What drugs does the ship\'s medicine chest have for suspected ACS, and what doses should be given?', 'domain': 'medical', 'topic': 'cardiac'},
    {'question': 'What are the WHO IMGS signs distinguishing a tension pneumothorax from a simple pneumothorax, and what emergency intervention is indicated?', 'domain': 'medical', 'topic': 'chest'},
    {'question': 'A crew member returns from shore leave in a tropical port with fever, rigors, and headache 10 days later. What is the most likely diagnosis and what WHO IMGS guidance applies?', 'domain': 'medical', 'topic': 'tropical medicine'},
    {'question': 'What is the WHO IMGS definition of a medical emergency requiring radio medical advice, and what information should the MPIC prepare before calling?', 'domain': 'medical', 'topic': 'telemedicine'},
    {'question': 'A crew member who is 36 weeks pregnant goes into labour at sea. What immediate steps should the MPIC take and what equipment will be needed?', 'domain': 'medical', 'topic': 'obstetrics'},
    {'question': 'A crew member is exhibiting acute psychosis — disorganised speech and aggression. What pharmacological intervention does the WHO IMGS recommend and what dose?', 'domain': 'medical', 'topic': 'mental health'},
    {'question': 'According to the WHO IMGS anatomy chapter, where is the liver located relative to the diaphragm and what physical signs suggest hepatomegaly?', 'domain': 'medical', 'topic': 'anatomy'},
    {'question': 'A crew member accidentally ingested a cleaning product containing sodium hydroxide (lye). Should vomiting be induced? What does WHO IMGS recommend?', 'domain': 'medical', 'topic': 'poisoning'},
    {'question': 'A welder\'s assistant has a chemical splash to both eyes. Describe the WHO IMGS irrigation procedure and how long it should continue.', 'domain': 'medical', 'topic': 'eye injuries'},
    {'question': 'What vital signs should the MPIC record for a seriously ill patient, and at what frequency does the WHO IMGS recommend recording them?', 'domain': 'medical', 'topic': 'assessment'},
    {'question': 'A crew member has severe toothache at sea with signs of dental abscess. What antibiotic and analgesic regimen does the WHO IMGS recommend?', 'domain': 'medical', 'topic': 'dental'},
    {'question': 'A sailor pulled from cold water has a core temperature of 30°C and is unconscious. What does the WHO IMGS say about rewarming technique and CPR in severe hypothermia?', 'domain': 'medical', 'topic': 'hypothermia'},
]

GENERAL_QUESTIONS = [
    {'question': 'What are the SOLAS requirements for the number of survival craft a cargo ship must carry?', 'domain': 'general', 'topic': 'SOLAS'},
    {'question': 'Explain how a centrifugal pump primes itself and what causes cavitation.', 'domain': 'general', 'topic': 'engineering'},
    {'question': 'What is the difference between a four-stroke and two-stroke diesel engine in terms of power stroke frequency?', 'domain': 'general', 'topic': 'engineering'},
    {'question': 'What does ISM Code require the master to do after a near-miss incident?', 'domain': 'general', 'topic': 'safety management'},
    {'question': 'Describe the fire triangle and explain how CO2 extinguishers suppress fire.', 'domain': 'general', 'topic': 'fire safety'},
    {'question': 'What is the purpose of a bilge alarm and at what level does it typically activate?', 'domain': 'general', 'topic': 'engineering'},
    {'question': 'What navigational lights must a vessel under 50 metres show when underway at night?', 'domain': 'general', 'topic': 'navigation'},
    {'question': 'What is a Mayday call and what information must it contain according to GMDSS procedure?', 'domain': 'general', 'topic': 'communications'},
    {'question': 'Explain the difference between true wind and apparent wind.', 'domain': 'general', 'topic': 'sailing'},
    {'question': 'What is cathodic protection and why is it used on ship hulls?', 'domain': 'general', 'topic': 'engineering'},
]

ALL_QUESTIONS = MEDICAL_QUESTIONS + GENERAL_QUESTIONS
print(f'{len(MEDICAL_QUESTIONS)} medical + {len(GENERAL_QUESTIONS)} general = {len(ALL_QUESTIONS)} total questions')

20 medical + 10 general = 30 total questions


In [5]:
import asyncio, json, time
import httpx

OLLAMA_URL = 'http://localhost:11434'
# Judge with Gemma, not Gemini. The app's cloud mode uses this exact model.
# Gemma-with-Gemma keeps the eval on-brand for the Gemma hackathon and makes
# the comparison story cleaner: "we ran a maritime Q&A bake-off between two
# Gemmas, judged by a third Gemma."
JUDGE_MODEL = 'gemma-4-26b-a4b-it'

# Google AI Studio sometimes hangs on this endpoint even for short responses;
# 180s is high but leaves the eval forward-progressing instead of giving up.
JUDGE_TIMEOUT = 180.0

# Truncate the response sent to the judge — keeps judge calls fast and is
# fair because the substance of every WHO IMGS answer is in the opening
# paragraph (drug, dose, route, duration). Trailing prose is filler.
JUDGE_RESPONSE_LIMIT = 1500

MEDICAL_SYSTEM = (
    'You are a maritime medical assistant trained on the WHO International Medical '
    'Guide for Ships (3rd Edition). Provide accurate, actionable guidance for '
    'emergencies at sea. Cite the relevant WHO IMGS page number when you reference '
    'a specific protocol or dosage.'
)

JUDGE_PROMPT = """Score this AI response to a maritime question on four axes (1-10).

Question: {question}
Domain: {domain} ({topic})

Response to evaluate:
---
{response}
---

Axes:
- accuracy: factually correct for maritime/medical use
- specificity: gives concrete doses, quantities, procedures
- citation: cites WHO IMGS page numbers or named protocols (medical only)
- hallucination: 10 = no invented numbers, 1 = serious fabrications

Return a JSON object with keys: accuracy, specificity, citation, hallucination, reasoning."""

JUDGE_SCHEMA = {
    'type': 'object',
    'properties': {
        'accuracy':      {'type': 'integer', 'minimum': 1, 'maximum': 10},
        'specificity':   {'type': 'integer', 'minimum': 1, 'maximum': 10},
        'citation':      {'type': 'integer', 'minimum': 1, 'maximum': 10},
        'hallucination': {'type': 'integer', 'minimum': 1, 'maximum': 10},
        'reasoning':     {'type': 'string'},
    },
    'required': ['accuracy', 'specificity', 'citation', 'hallucination', 'reasoning'],
}


async def ollama_chat(model, system, prompt, timeout=120.0, max_retries=3):
    payload = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': prompt},
        ],
        'stream': False,
        'options': {'temperature': 0.1, 'num_predict': 512},
    }
    delay = 3.0
    for attempt in range(max_retries):
        try:
            async with httpx.AsyncClient(timeout=timeout) as client:
                r = await client.post(f'{OLLAMA_URL}/api/chat', json=payload)
                r.raise_for_status()
                return r.json()['message']['content']
        except (httpx.ConnectError, httpx.RemoteProtocolError) as e:
            if attempt < max_retries - 1:
                print(f'    [ollama connect error, retrying in {delay:.0f}s: {e}]')
                await asyncio.sleep(delay)
                delay *= 2
            else:
                raise


def _extract_json(text: str):
    """Pull the first {...} block out of the model's reply and parse it.

    With responseMimeType=application/json the model SHOULD return raw JSON,
    but this is a safety net for any preamble/fence the API lets through.
    """
    if not text or not text.strip():
        raise ValueError('empty response')
    stripped = text.strip()
    if stripped.startswith('{') and stripped.endswith('}'):
        return json.loads(stripped)
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f'no JSON object in: {text[:120]!r}')
    return json.loads(text[start:end+1])


# Free-tier judging is rate-limited. Retry on 429 with exponential backoff;
# fall through to None if we keep getting throttled so the eval still completes.
async def gemma_judge(question, domain, topic, response, max_retries=5):
    # Truncate long responses — keeps judging fast and avoids ReadTimeouts.
    if len(response) > JUDGE_RESPONSE_LIMIT:
        response = response[:JUDGE_RESPONSE_LIMIT] + '\n...[truncated]'

    prompt = JUDGE_PROMPT.format(question=question, domain=domain, topic=topic, response=response)
    payload = {
        'contents': [{'parts': [{'text': prompt}]}],
        'generationConfig': {
            'temperature': 0.0,
            'maxOutputTokens': 512,
            # JSON mode — forces the decoder to emit only valid JSON, no
            # chain-of-thought scratchpad.
            'responseMimeType': 'application/json',
            'responseSchema': JUDGE_SCHEMA,
        },
    }
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{JUDGE_MODEL}:generateContent?key={GOOGLE_API_KEY}'

    delay = 4.0
    last_text = None
    for attempt in range(max_retries):
        try:
            async with httpx.AsyncClient(timeout=JUDGE_TIMEOUT) as client:
                r = await client.post(url, json=payload)

            if r.status_code == 429:
                wait = float(r.headers.get('retry-after', delay))
                print(f'    [judge throttled, sleeping {wait:.1f}s — attempt {attempt+1}/{max_retries}]')
                await asyncio.sleep(wait)
                delay = min(delay * 2, 60)
                continue

            if r.status_code >= 400:
                if attempt == 0 and 'responseSchema' in r.text:
                    print(f'    [judge: responseSchema not supported, retrying without schema]')
                    payload['generationConfig'].pop('responseSchema', None)
                    continue
                print(f'    [judge HTTP {r.status_code}: {r.text[:240]}]')
                await asyncio.sleep(delay)
                delay = min(delay * 2, 60)
                continue

            data = r.json()
            try:
                last_text = data['candidates'][0]['content']['parts'][0]['text']
            except (KeyError, IndexError):
                print(f'    [judge: no text in response: {json.dumps(data)[:240]}]')
                await asyncio.sleep(delay)
                delay = min(delay * 2, 60)
                continue
            return _extract_json(last_text)
        except Exception as e:
            snippet = (last_text or '')[:160]
            print(f'    [judge error attempt {attempt+1}: {type(e).__name__}: {e} | text={snippet!r}]')
            await asyncio.sleep(delay)
            delay = min(delay * 2, 60)

    return None


# Back-compat alias
gemini_judge = gemma_judge


def composite(scores):
    if not scores: return None
    return round(sum(scores[k] for k in ('accuracy','specificity','citation','hallucination')) / 4, 1)

print('Functions defined. Judge:', JUDGE_MODEL, '(JSON mode, timeout', JUDGE_TIMEOUT, 's, response cap', JUDGE_RESPONSE_LIMIT, 'chars)')

Functions defined. Judge: gemma-4-26b-a4b-it (JSON mode, timeout 180.0 s, response cap 1500 chars)


In [6]:
results = []

async def run_all():
    for i, q in enumerate(ALL_QUESTIONS, 1):
        question, domain, topic = q['question'], q['domain'], q['topic']
        system = MEDICAL_SYSTEM if domain == 'medical' else 'You are a knowledgeable maritime professional.'
        print(f'[{i:2d}/{len(ALL_QUESTIONS)}] {domain:8s} / {topic}')
        print(f'         Q: {question[:75]}...')

        row = {'question': question, 'domain': domain, 'topic': topic}

        for label, model in [('baseline', BASELINE_MODEL), ('finetuned', MEDICAL_MODEL)]:
            t0 = time.monotonic()
            try:
                response = await ollama_chat(model, system, question)
                elapsed = time.monotonic() - t0
                print(f'         {label:10s}: {len(response)} chars in {elapsed:.1f}s')
            except Exception as e:
                response = f'[ERROR: {e}]'
                elapsed = 0
                print(f'         {label:10s}: ERROR — {e}')

            row[f'{label}_response'] = response
            row[f'{label}_latency']  = round(elapsed, 2)

            scores = await gemini_judge(question, domain, topic, response)
            row[f'{label}_scores'] = scores
            if scores:
                avg = composite(scores)
                print(f'         {label:10s}: judge avg={avg}  ({scores.get("reasoning","")[:60]})')
            await asyncio.sleep(0.5)

        results.append(row)
        print()

await run_all()
print('\nEval complete!')

[ 1/30] medical  / pharmacology
         Q: What is the recommended adult oral dose of amoxicillin for a chest infectio...
         baseline  : ERROR — 
         baseline  : judge avg=1.0  (The response provided is an error message '[ERROR: ]' rather)
         finetuned : 1879 chars in 23.0s
    [judge error attempt 1: ReadTimeout:  | text='']
    [judge error attempt 2: ReadTimeout:  | text='']
    [judge error attempt 3: ReadTimeout:  | text='']
    [judge error attempt 4: ReadTimeout:  | text='']
    [judge error attempt 5: ReadTimeout:  | text='']

[ 2/30] medical  / envenomation
         Q: A sailor has been stung by a jellyfish and is experiencing urticaria. What ...
         baseline  : 431 chars in 13.5s
         baseline  : judge avg=3.2  (The response is incomplete and cut off mid-sentence, providi)
         finetuned : 1031 chars in 7.6s
         finetuned : judge avg=9.2  (The response provides correct medical guidance for urticaria)

[ 3/30] medical  / procedures
         

In [7]:
from IPython.display import Markdown, display

medical  = [r for r in results if r['domain'] == 'medical']
general  = [r for r in results if r['domain'] == 'general']

def avg_score(rows, label):
    s = [composite(r.get(f'{label}_scores')) for r in rows]
    s = [x for x in s if x is not None]
    return f'{sum(s)/len(s):.1f}' if s else 'n/a'

def avg_lat(rows, label):
    lats = [r.get(f'{label}_latency', 0) for r in rows]
    return f'{sum(lats)/len(lats):.1f}s'

lines = [
    '## Eval Results: Unsloth Fine-tune vs Vanilla Gemma 4',
    '',
    f'**Baseline:** `{BASELINE_MODEL}`  ',
    f'**Fine-tune:** `{MEDICAL_MODEL}`  ',
    '**Scoring:** Gemma-as-judge (`gemma-4-26b-a4b-it`) — 1–10 per axis: accuracy, specificity, citation, anti-hallucination  ',
    f'**Questions:** {len(medical)} held-out WHO IMGS medical + {len(general)} general maritime (regression check)',
    '',
    '### Summary',
    '',
    '| Category | N | Baseline avg | Fine-tune avg | Baseline latency | Fine-tune latency |',
    '|----------|---|-------------|---------------|-----------------|-------------------|',
    f'| Medical (WHO IMGS) | {len(medical)} | {avg_score(medical,"baseline")} | {avg_score(medical,"finetuned")} | {avg_lat(medical,"baseline")} | {avg_lat(medical,"finetuned")} |',
    f'| General maritime   | {len(general)} | {avg_score(general,"baseline")} | {avg_score(general,"finetuned")} | {avg_lat(general,"baseline")} | {avg_lat(general,"finetuned")} |',
    '',
    '> General maritime questions are a **regression check** — a large drop on the fine-tune would indicate harm to general capability.',
    '',
    '### Per-question breakdown (medical)',
    '',
    '| Topic | Question | Baseline | Fine-tune | Δ |',
    '|-------|----------|----------|-----------|---|',
]

for r in medical:
    b = composite(r.get('baseline_scores'))
    f = composite(r.get('finetuned_scores'))
    delta = f'**+{f-b:.1f}**' if (b and f and f > b) else (f'{f-b:.1f}' if (b and f) else 'n/a')
    lines.append(f'| {r["topic"]} | {r["question"][:60]}… | {b or "n/a"} | {f or "n/a"} | {delta} |')

lines += [
    '',
    '### Per-question breakdown (general maritime — regression check)',
    '',
    '| Topic | Question | Baseline | Fine-tune | Δ |',
    '|-------|----------|----------|-----------|---|',
]

for r in general:
    b = composite(r.get('baseline_scores'))
    f = composite(r.get('finetuned_scores'))
    delta = f'**+{f-b:.1f}**' if (b and f and f > b) else (f'{f-b:.1f}' if (b and f) else 'n/a')
    lines.append(f'| {r["topic"]} | {r["question"][:60]}… | {b or "n/a"} | {f or "n/a"} | {delta} |')

report = '\n'.join(lines)
display(Markdown(report))

# Save to file
with open('/kaggle/working/eval_report.md', 'w') as f:
    f.write(report)
with open('/kaggle/working/eval_results.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')

print('\nSaved eval_report.md and eval_results.jsonl to /kaggle/working/')

## Eval Results: Unsloth Fine-tune vs Vanilla Gemma 4

**Baseline:** `gemma4:e2b`  
**Fine-tune:** `hf.co/nswitzer/gemma4-maritime-medical-GGUF`  
**Scoring:** Gemma-as-judge (`gemma-4-26b-a4b-it`) — 1–10 per axis: accuracy, specificity, citation, anti-hallucination  
**Questions:** 20 held-out WHO IMGS medical + 10 general maritime (regression check)

### Summary

| Category | N | Baseline avg | Fine-tune avg | Baseline latency | Fine-tune latency |
|----------|---|-------------|---------------|-----------------|-------------------|
| Medical (WHO IMGS) | 20 | 3.6 | 5.8 | 11.2s | 9.8s |
| General maritime   | 10 | 6.4 | 6.0 | 12.5s | 10.3s |

> General maritime questions are a **regression check** — a large drop on the fine-tune would indicate harm to general capability.

### Per-question breakdown (medical)

| Topic | Question | Baseline | Fine-tune | Δ |
|-------|----------|----------|-----------|---|
| pharmacology | What is the recommended adult oral dose of amoxicillin for a… | 1.0 | n/a | n/a |
| envenomation | A sailor has been stung by a jellyfish and is experiencing u… | 3.2 | 9.2 | **+6.0** |
| procedures | The MPIC needs to give an intramuscular injection. Describe … | 4.2 | n/a | n/a |
| pharmacology | A crew member has been prescribed metronidazole. What food i… | 5.8 | 6.2 | **+0.4** |
| surgical emergencies | What are the WHO IMGS criteria for deciding to evacuate a pa… | 3.2 | n/a | n/a |
| burns | How do you estimate the percentage body surface area burned … | n/a | 2.5 | n/a |
| burns | A crew member has a deep second-degree burn covering approxi… | n/a | 3.0 | n/a |
| wounds | Describe the WHO IMGS approach to wound closure for a lacera… | 1.0 | n/a | n/a |
| cardiac | A 52-year-old Chief Engineer has crushing central chest pain… | n/a | n/a | n/a |
| chest | What are the WHO IMGS signs distinguishing a tension pneumot… | 5.5 | n/a | n/a |
| tropical medicine | A crew member returns from shore leave in a tropical port wi… | 5.5 | 7.2 | **+1.7** |
| telemedicine | What is the WHO IMGS definition of a medical emergency requi… | n/a | 3.0 | n/a |
| obstetrics | A crew member who is 36 weeks pregnant goes into labour at s… | n/a | n/a | n/a |
| mental health | A crew member is exhibiting acute psychosis — disorganised s… | 3.2 | 7.5 | **+4.3** |
| anatomy | According to the WHO IMGS anatomy chapter, where is the live… | 4.2 | n/a | n/a |
| poisoning | A crew member accidentally ingested a cleaning product conta… | 5.8 | 7.5 | **+1.7** |
| eye injuries | A welder's assistant has a chemical splash to both eyes. Des… | 6.0 | 6.5 | **+0.5** |
| assessment | What vital signs should the MPIC record for a seriously ill … | n/a | n/a | n/a |
| dental | A crew member has severe toothache at sea with signs of dent… | 1.0 | n/a | n/a |
| hypothermia | A sailor pulled from cold water has a core temperature of 30… | 1.0 | n/a | n/a |

### Per-question breakdown (general maritime — regression check)

| Topic | Question | Baseline | Fine-tune | Δ |
|-------|----------|----------|-----------|---|
| SOLAS | What are the SOLAS requirements for the number of survival c… | 7.8 | n/a | n/a |
| engineering | Explain how a centrifugal pump primes itself and what causes… | n/a | n/a | n/a |
| engineering | What is the difference between a four-stroke and two-stroke … | n/a | 6.8 | n/a |
| safety management | What does ISM Code require the master to do after a near-mis… | 5.5 | n/a | n/a |
| fire safety | Describe the fire triangle and explain how CO2 extinguishers… | 5.5 | n/a | n/a |
| engineering | What is the purpose of a bilge alarm and at what level does … | n/a | n/a | n/a |
| navigation | What navigational lights must a vessel under 50 metres show … | 7.8 | 2.8 | -5.0 |
| communications | What is a Mayday call and what information must it contain a… | 5.5 | 6.5 | **+1.0** |
| sailing | Explain the difference between true wind and apparent wind.… | n/a | 6.8 | n/a |
| engineering | What is cathodic protection and why is it used on ship hulls… | n/a | 7.0 | n/a |


Saved eval_report.md and eval_results.jsonl to /kaggle/working/
